<a href="https://colab.research.google.com/github/haga-kazutoshi/-15G/blob/main/20260701_%E8%8A%B3%E8%B3%80_%E3%83%AD%E3%82%B8%E3%82%B9%E3%83%86%E3%82%A3%E3%83%83%E3%82%AF%E5%9B%9E%E5%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. ロジスティック回帰とは
目的変数が0と1からなる2値のデータ、あるいは0から1までの値からなる確率などのデータについて、説明変数を使った式で表す方法

→ ある事象が「起きるか,起きないか」(0か1か) または「ある事象が起こる確率」を2つのグループに分類 (予測) を行う

## メリット
*    計算が軽くて速い：シンプルな仕組みのため, データが大量でも計算が速い.

*    結果の理由が分かりやすい：どのデータが予測に影響を与えたかが解釈しやすい.

*    様々な問題に適応できる：モデルに汎用性があるため, 複雑な関係性を取り込める.



# 1. 乳がんのデータセットを分析

In [ ]:
# 1. 必要なライブラリのインポート
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer #
from sklearn.model_selection import train_test_split #
from sklearn.preprocessing import StandardScaler #
from sklearn.linear_model import LogisticRegression #
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #

4 `from sklearn.datasets import load_breast_cancer`
* `sklearn.datasets`：scikit-learnが用意してくれている、練習用の有名なデータセット
* `load_breast_cancer`：乳がんのデータセットを読み込むための機能

5 `from sklearn.model_selection import train_test_split`
* `sklearn.model_selection`：データを分割、モデルの検証方法を決定など
* `train_test_split`：データを「訓練用」と「テスト用」に分割する機能

6 `from sklearn.preprocessing import StandardScaler`
* `sklearn.preprocessing`：データの前処理・加工を行う
* `StandardScaler`：データを標準化する機能

7 `from sklearn.linear_model import LogisticRegression`
* `sklearn.linear_model`：線形モデルと呼ばれる、直線や平面を使った計算アルゴリズムなど
* `LogisticRegression`：ロジスティック回帰の本体

8 `from sklearn.metrics import accuracy_score, classification_report, confusion_matrix`
* `sklearn.metrics`：モデルの予測結果がどれくらい正しかったかを測る、評価用のモデル
* `accuracy_score`：正解率（全体のうち、何％の予測が当たっていたか）を計算
* `classification_report`：正解率だけでなく、見落としがなかったか（再現率）や、空振りがなかったか（適合率）などをまとめて見やすい表にする
* `confusion_matrix`：混同行列と呼ばれる、「本当は悪性なのに良性と間違えた数」や「良性を正しく良性と当てられた数」をタテヨコの表（マトリクス）にしてくれる


In [ ]:
# 2. データの読み込みと準備
cancer = load_breast_cancer() # 乳がんのデータをダウンロード
X = pd.DataFrame(cancer.data, columns=cancer.feature_names) #データフレーム化
y = cancer.target  # 0: 悪性(Malignant), 1: 良性(Benign)　正解のデータを抜き出す

# 分かりやすくするために、特徴量を2つだけに絞ります（細胞の平均半径と平均凹み）
X_selected = X[['mean radius', 'mean concavity']]

# 3. データを訓練用とテスト用に分割 (80% を訓練用、20% をテスト用)
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

2 `cancer = load_breast_cancer()`：乳がんのデータをダウンロード

3 `X = pd.DataFrame(cancer.data, columns=cancer.feature_names)`：データフレーム化

4 `y = cancer.target`：正解のデータを抜き出す

10～12
```
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)
```
データ (`X_selected`) と正解 (`y`) を訓練用 (80%) とテスト用 (20%) の合計4つのグループに分割

* `X_selected, y`：分割したい元のデータを指定
* `test_size=0.2`：全体の20%をテスト用にする。残り80%を訓練用にする
* `random_state=42`：データをランダムに分ける際の乱数のシードを固定
* `stratify=y`：元のデータにある「良性」と「悪性」の比率を崩さず割り振る

作成した4つの変数
* `X_train`：訓練用のデータ（手がかり・全体の80%）
* `X_test`：テスト用のデータ（手がかり・全体の20%）
* `y_train`：訓練用の正解（良性か悪性か・全体の80%）
* `y_test`：テスト用の正解（良性か悪性か・全体の20%）

In [ ]:
# 4. データの標準化（ロジスティック回帰の性能を上げるための大切なステップ）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. ロジスティック回帰モデルの作成と学習
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# 6. テストデータを使って予測
y_pred = model.predict(X_test_scaled)

2 `scaler = StandardScaler()`：データを標準化 (平均0, 標準偏差1に変換)

3 `X_train_scaled = scaler.fit_transform(X_train)`
* `fit`：訓練データ(`X_train`) の平均値と標準偏差を計算し記憶
* `transform`：データを標準化 (平均0, 標準偏差1に変換)

4 `X_test_scaled = scaler.transform(X_test)`：テストデータも訓練データと全く同じく標準化

7 `model = LogisticRegression()`：ロジスティック回帰のモデルを用意

8 `model.fit(X_train_scaled, y_train)`：訓練データ (`X_train_scaled`) と正解 (`y_train`) を使って学習

11 `y_pred = model.predict(X_test_scaled)`：テストを解いて予測結果を出力



In [ ]:
# 7. 結果の評価
print("=== モデルの評価結果 ===")
print(f"正解率 (Accuracy): {accuracy_score(y_test, y_pred):.2f}")
print("\n--- 詳細なレポート ---")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

=== モデルの評価結果 ===
正解率 (Accuracy): 0.92

--- 詳細なレポート ---
              precision    recall  f1-score   support

   malignant       0.88      0.90      0.89        42
      benign       0.94      0.93      0.94        72

    accuracy                           0.92       114
   macro avg       0.91      0.92      0.92       114
weighted avg       0.92      0.92      0.92       114



3 `{accuracy_score(y_test, y_pred):.2f}")`：予測の精度を計算
* `y_test`：本物の正解
* `y_pred`：予測結果

5 `classification_report(y_test, y_pred, target_names=cancer.target_names)`：正解率だけでは見抜けない、予測の癖をまとめたレポートを出力
* `y_test`：本物の正解
* `y_pred`：予測結果
* `target_names=cancer.target_names`：表の行に0や1ではなく、`malignant`(悪性)、`benign`(良性) と言葉で表現

### 詳細レポートの読み方

それぞれの列の意味
* `precision`：適合率、Aと予測したもののうち、本当にAだった確率
* `recall`：再現率、Aのデータのうち、予測が漏らさずにAを見つけられた確率
* `f1-score`：F1値、`precision`と`recall`の平均
* `support`：データの個数

それぞれの行の意味
* `malignant`：悪性
* `benign`：良性

* `accuracy`：全体の正解率
* `macro avg`：マクロ平均、悪性と良性の数値の平均値
* `weighted avg`：加重平均、データの数を考慮して重み付けした平均値